# T2.5 Analyze the models with respect to different groups

In [29]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from collections.abc import Callable

from source.metrics.util import RecommenderDataLoader
from source.metrics.beyond_accuracy import BeyondAccuracyMetrics, jaccard_distance
from source.metrics.evaluation_metrics import EvaluationMetrics
from source.metrics.ranking_metrics import RankingMetrics


In [3]:
# Config
MODELS = ["EASE", "BPR"]

In [4]:
def load_evaluation_data(prediction_file: str):
    """
    Helper funtion to load scores, align IDs, and build boolean matrices.
    Returns: (user_id2idx, item_id2idx, predicted_scores, train_mask, test_mask, validation_mask)
    """
    with RecommenderDataLoader(base_dir="../../results") as loader:
        users, items, predicted_scores = loader.load_predictions(npz_filename=prediction_file)
        num_users, num_items = predicted_scores.shape

        user_id2idx = {int(raw_id): idx for idx, raw_id in enumerate(users)}
        item_id2idx = {int(raw_id): idx for idx, raw_id in enumerate(items)}

        def build_dense_mask(filename: str) -> np.ndarray:
            mask = np.zeros((num_users, num_items), dtype=bool)
            try:
                pairs = loader.load_split_data_file(filename)
                for row in pairs:
                    if len(row) >= 2:
                        raw_u, raw_i = int(row[0]), int(row[1])
                        if raw_u in user_id2idx and raw_i in item_id2idx:
                            mask[user_id2idx[raw_u], item_id2idx[raw_i]] = True
            except (FileNotFoundError, AssertionError) as e:
                print(f"Warning: {filename} skipped or failed. {e}")
            return mask

        train_mask = build_dense_mask("train.tsv")
        test_mask = build_dense_mask("test.tsv")
        validation_mask = build_dense_mask("valid.tsv")

    return user_id2idx, item_id2idx, predicted_scores, train_mask, test_mask, validation_mask


def build_wide_evaluation_dataframe_v2(
    model_names: list[str],
    loader_func: Callable,
    user_info: dict[int, tuple[int, str, str, str]],
    item_info: dict[int, tuple[str, int, list[str]]],
    ground_truth: dict[tuple[int, int], float],
) -> pd.DataFrame:
    """
    Build one row per (user_id, item_id) pair.

    Columns include:
      - user_id, item_id
      - data_split_type
      - one prediction column per model
      - ground_truth
      - user and item metadata

    Predictions and masks are aligned by raw IDs, not assumed
    to have identical positional ordering across models.
    """
    if not model_names:
        raise ValueError("model_names must contain at least one model.")

    df = None
    base_users = None
    base_items = None
    base_split = None

    for model_name in model_names:
        filename = f"{model_name}.npz"

        user_id2idx, item_id2idx, predicted_scores, train_mask, test_mask, validation_mask = loader_func(filename)

        predicted_scores = np.asarray(predicted_scores)
        train_mask = np.asarray(train_mask, dtype=bool)
        test_mask = np.asarray(test_mask, dtype=bool)
        validation_mask = np.asarray(validation_mask, dtype=bool)

        num_users, num_items = predicted_scores.shape
        expected_shape = (num_users, num_items)
        if train_mask.shape != expected_shape or test_mask.shape != expected_shape or validation_mask.shape != expected_shape:
            raise ValueError(f"{model_name}: prediction and mask shapes do not match.")

        # Recover raw IDs in their actual matrix-index order.
        def ids_in_index_order(id2idx, expected_length, name):
            ids = np.empty(expected_length, dtype=np.int64)
            seen = np.zeros(expected_length, dtype=bool)

            for raw_id, idx in id2idx.items():
                idx = int(idx)
                if idx < 0 or idx >= expected_length or seen[idx]:
                    raise ValueError(f"{model_name}: invalid {name} index mapping.")  # noqa: B023
                ids[idx] = int(raw_id)
                seen[idx] = True

            if not seen.all() or len(id2idx) != expected_length:
                raise ValueError(f"{model_name}: {name} mapping is incomplete.")  # noqa: B023

            return ids

        users = ids_in_index_order(user_id2idx, num_users, "user")
        items = ids_in_index_order(item_id2idx, num_items, "item")

        # A pair must not belong to multiple splits.
        overlap = (train_mask & test_mask) | (train_mask & validation_mask) | (test_mask & validation_mask)
        if overlap.any():
            raise ValueError(f"{model_name}: train/test/validation masks overlap for {int(overlap.sum())} user-item pairs.")

        if df is None:
            # Construct the master grid in the first model's ID order.
            base_users = users.copy()
            base_items = items.copy()

            u_idx, i_idx = np.indices(expected_shape)

            df = pd.DataFrame({"user_id": users[u_idx.ravel()], "item_id": items[i_idx.ravel()]})
            split = np.full(expected_shape, "unassigned", dtype=object)
            split[train_mask] = "train"
            split[test_mask] = "test"
            split[validation_mask] = "validation"
            base_split = split.ravel()
            df["data_split_type"] = base_split

        else:
            # Verify that all models cover the same ID sets.
            assert base_users is not None
            assert base_items is not None
            if set(users) != set(base_users) or set(items) != set(base_items):
                raise ValueError(f"{model_name}: user/item ID sets differ from the first model.")

            # Convert the master grid's ID order to this model's
            # positional order before flattening predictions.
            user_positions = np.array([user_id2idx[int(uid)] for uid in base_users])
            item_positions = np.array([item_id2idx[int(iid)] for iid in base_items])

            predicted_scores = predicted_scores[np.ix_(user_positions, item_positions)]

            current_split = np.full(expected_shape, "unassigned", dtype=object)
            current_split[train_mask] = "train"
            current_split[test_mask] = "test"
            current_split[validation_mask] = "validation"

            current_split = current_split[np.ix_(user_positions, item_positions)].ravel()

            assert base_split is not None
            if not np.array_equal(current_split, base_split):
                raise ValueError(f"{model_name}: split assignments differ from the first model after ID alignment.")

        # Predictions now follow the master dataframe's row order.
        df[model_name] = predicted_scores.ravel().astype(np.float32)

    assert df is not None

    # Map ground truth using the exact (user_id, item_id) pair.
    gt = pd.Series(ground_truth, name="ground_truth", dtype=np.float32)
    pair_index = pd.MultiIndex.from_arrays([df["user_id"], df["item_id"]])
    df["ground_truth"] = gt.reindex(pair_index).to_numpy()

    # Join user metadata.
    user_meta = pd.DataFrame.from_dict(
        user_info,
        orient="index",
        columns=["age", "gender", "occupation", "zip_code"],
    )
    user_meta.index = user_meta.index.astype("int64")
    user_meta.index.name = "user_id"

    # Join item metadata.
    item_meta = pd.DataFrame.from_dict(
        item_info,
        orient="index",
        columns=["title", "year", "genres"],
    )
    item_meta.index = item_meta.index.astype("int64")
    item_meta.index.name = "item_id"

    df = df.merge(user_meta.reset_index(), on="user_id", how="left", validate="many_to_one")
    df = df.merge(item_meta.reset_index(), on="item_id", how="left", validate="many_to_one")

    # Add age groups
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 18, 26, 36, 46, 56, 101],
        labels=["<18", "18-25", "26-35", "36-45", "46-55", "56+"],
        right=False,
    )

    for col in ["gender", "occupation", "age_group", "data_split_type"]:
        df[col] = df[col].astype("category")

    # Final integrity checks.
    if df.duplicated(["user_id", "item_id"]).any():
        raise AssertionError("Duplicate user-item pairs found.")

    assert base_users is not None
    assert base_items is not None
    expected_rows = len(base_users) * len(base_items)
    if len(df) != expected_rows:
        raise AssertionError(f"Expected {expected_rows} rows, got {len(df)}.")

    return df

In [17]:
df = (
    build_wide_evaluation_dataframe_v2(
        model_names=MODELS,
        loader_func=load_evaluation_data,
        user_info=RecommenderDataLoader().load_user_info(file_path=Path("..", "..", "data", "ml-100k", "ml-100k.user")),
        item_info=RecommenderDataLoader().load_item_info(file_path=Path("..", "..", "data", "ml-100k", "ml-100k.item")),
        ground_truth=RecommenderDataLoader().load_interaction_ratings(file_path=Path("..", "..", "data", "ml-100k", "ml-100k.inter")),
    )
    # .dropna(subset=["ground_truth"])
    # .reset_index(drop=True)
)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1586126 entries, 0 to 1586125
Data columns (total 14 columns):
 #   Column           Non-Null Count    Dtype   
---  ------           --------------    -----   
 0   user_id          1586126 non-null  int64   
 1   item_id          1586126 non-null  int64   
 2   data_split_type  1586126 non-null  category
 3   EASE             1586126 non-null  float32 
 4   BPR              1586126 non-null  float32 
 5   ground_truth     100000 non-null   float32 
 6   age              1586126 non-null  int64   
 7   gender           1586126 non-null  category
 8   occupation       1586126 non-null  category
 9   zip_code         1586126 non-null  object  
 10  title            1586126 non-null  object  
 11  year             1586126 non-null  int64   
 12  genres           1586126 non-null  object  
 13  age_group        1586126 non-null  category
dtypes: category(4), float32(3), int64(4), object(3)
memory usage: 108.9+ MB


In [ ]:
display(df.groupby("user_id").size().describe())

count    943.000000
mean     106.044539
std      100.931743
min       20.000000
25%       33.000000
50%       65.000000
75%      148.000000
max      737.000000
dtype: float64

In [33]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

MODELS = ["EASE", "BPR"]
K = 10
ALPHA = 0.05  # FDR level used when flagging cells

HIDDEN_SPLITS = ("train", "validation")  # never re-recommended
TEST_SPLIT = "test"                      # "unassigned" pairs remain candidates

ACTIVITY_LABELS = ["low", "mid-low", "mid-high", "high"]
POPULARITY_LABELS = ["long-tail", "low-mid", "high-mid", "head"]
USER_DIMS = ["age_group", "gender", "occupation", "activity"]

# dimension -> {group label -> boolean mask over matrix rows (users) or columns (items)}
GroupMasks = dict[str, dict[str, np.ndarray]]


@dataclass(frozen=True)
class EvaluationData:
    scores: dict[str, np.ndarray]  # model -> (n_users, n_items)
    hidden: np.ndarray             # (n_users, n_items) bool
    relevant: np.ndarray           # (n_users, n_items) bool
    users: pd.DataFrame            # attributes, row-aligned with the matrices
    items: pd.DataFrame            # attributes, column-aligned with the matrices


def build_evaluation_data(df: pd.DataFrame, models: list[str]) -> EvaluationData:
    """Turn the long (user, item) grid dataframe into aligned matrices."""
    df = df.sort_values(["user_id", "item_id"]).reset_index(drop=True)
    user_ids = np.sort(df["user_id"].unique())
    item_ids = np.sort(df["item_id"].unique())
    shape = (len(user_ids), len(item_ids))
    if len(df) != shape[0] * shape[1]:
        raise ValueError("df is not the full user x item grid.")

    split = df["data_split_type"].astype(str).to_numpy().reshape(shape)
    return EvaluationData(
        scores={m: df[m].to_numpy().reshape(shape) for m in models},
        hidden=np.isin(split, HIDDEN_SPLITS),
        relevant=split == TEST_SPLIT,
        users=df.drop_duplicates("user_id").set_index("user_id").loc[user_ids].copy(),
        items=df.drop_duplicates("item_id").set_index("item_id").loc[item_ids].copy(),
    )


def quartile_labels(values: np.ndarray, labels: list[str]) -> pd.Categorical:
    """Equal-sized buckets; ranking first so ties cannot break qcut."""
    ranks = pd.Series(values).rank(method="first").to_numpy()
    return pd.qcut(ranks, q=len(labels), labels=labels)


def masks_by_category(series: pd.Series) -> dict[str, np.ndarray]:
    """One boolean mask per group, in category order (or sorted order)."""
    if isinstance(series.dtype, pd.CategoricalDtype):
        labels = series.cat.categories
    else:
        labels = sorted(series.dropna().unique())
    values = series.to_numpy()
    return {str(label): values == label for label in labels}


def make_user_groups(data: EvaluationData) -> GroupMasks:
    users = data.users.copy()
    users["activity"] = quartile_labels(data.hidden.sum(axis=1), ACTIVITY_LABELS)
    return {dim: masks_by_category(users[dim]) for dim in USER_DIMS}


def make_item_groups(data: EvaluationData) -> GroupMasks:
    items = data.items.copy()  # `genres` must be a list of strings per item
    items["decade"] = (items["year"] // 10 * 10).astype(str) + "s"
    items["pop_bucket"] = quartile_labels(data.hidden.sum(axis=0), POPULARITY_LABELS)

    all_genres = sorted({genre for genres in items["genres"] for genre in genres})
    genre_masks = {g: items["genres"].apply(lambda genres, g=g: g in genres).to_numpy() for g in all_genres}

    return {
        "decade": masks_by_category(items["decade"]),
        "pop_bucket": masks_by_category(items["pop_bucket"]),
        "genre": genre_masks,
    }


def evaluate(
    data: EvaluationData,
    model: str,
    *,
    user_mask: np.ndarray | None = None,
    item_mask: np.ndarray | None = None,
    distance_matrix: np.ndarray | None = None,
    k: int = K,
) -> dict[str, float]:
    """
    Metrics for one model on a user and/or item group.

    Item groups only restrict which test items count as relevant; the ranking
    still runs over the full catalog. Beyond-accuracy metrics are computed
    only for user groups, and only if a distance matrix is passed.
    """
    scores, hidden, relevant = data.scores[model], data.hidden, data.relevant
    if item_mask is not None:
        relevant = relevant & item_mask[None, :]
    if user_mask is not None:
        scores, hidden, relevant = scores[user_mask], hidden[user_mask], relevant[user_mask]
    if not relevant.any():
        return {}

    metrics = RankingMetrics.calculate_all_metrics(scores, hidden, relevant, k)
    if distance_matrix is not None and item_mask is None:
        metrics |= BeyondAccuracyMetrics.calculate_all_metrics(scores, hidden, relevant, distance_matrix, k)
    return metrics


def user_group_table(data: EvaluationData, user_groups: GroupMasks, distance_matrix: np.ndarray, k: int = K) -> pd.DataFrame:
    rows = [
        {"model": model, "dim": dim, "group": group, "n_users": int(mask.sum()),
         **evaluate(data, model, user_mask=mask, distance_matrix=distance_matrix, k=k)}
        for model in data.scores
        for dim, groups in user_groups.items()
        for group, mask in groups.items()
    ]
    return pd.DataFrame(rows)


def item_group_table(data: EvaluationData, item_groups: GroupMasks, k: int = K) -> pd.DataFrame:
    rows = [
        {"model": model, "dim": dim, "group": group, "n_test": int((data.relevant & mask).sum()),
         **evaluate(data, model, item_mask=mask, k=k)}
        for model in data.scores
        for dim, groups in item_groups.items()
        for group, mask in groups.items()
    ]
    return pd.DataFrame(rows)

In [34]:
def per_user_metric(data: EvaluationData, model: str, metric: str, item_mask: np.ndarray, k: int = K) -> np.ndarray:
    relevant = data.relevant & item_mask[None, :]
    return RankingMetrics.calculate_per_user_metrics(data.scores[model], data.hidden, relevant, k)[metric]


def _flatten(groups: GroupMasks) -> list[tuple[str, str, np.ndarray]]:
    return [(dim, group, mask) for dim, by_group in groups.items() for group, mask in by_group.items()]


def screen_cells(
    data: EvaluationData,
    user_groups: GroupMasks,
    item_groups: GroupMasks,
    model_a: str,
    model_b: str,
    metric: str = f"recall@{K}",
    min_users: int = 20,
    min_test: int = 30,
    k: int = K,
) -> pd.DataFrame:
    """Evaluate every (user group, item group) cell, including the 'all' marginals."""
    n_users, n_items = data.relevant.shape
    user_groups = {"all": {"all": np.ones(n_users, dtype=bool)}, **user_groups}
    item_groups = {"all": {"all": np.ones(n_items, dtype=bool)}, **item_groups}

    # One ranking pass per (model, item group); each user group is then a cheap slice.
    per_user: dict[tuple, np.ndarray] = {}
    test_counts: dict[tuple, np.ndarray] = {}
    for idim, ig, imask in _flatten(item_groups):
        test_counts[idim, ig] = (data.relevant & imask[None, :]).sum(axis=1)
        for model in (model_a, model_b):
            per_user[model, idim, ig] = per_user_metric(data, model, metric, imask, k)

    rows = []
    for (udim, ug, umask), (idim, ig, _) in product(_flatten(user_groups), _flatten(item_groups)):
        a = per_user[model_a, idim, ig][umask]
        b = per_user[model_b, idim, ig][umask]
        valid = ~np.isnan(a)  # users with at least one test item in this cell
        n = int(valid.sum())
        n_test = int(test_counts[idim, ig][umask].sum())
        if n < min_users or n_test < min_test:
            continue

        diff = (a - b)[valid]
        se = diff.std(ddof=1) / np.sqrt(n)
        t = diff.mean() / se if se > 0 else 0.0
        rows.append({
            "user_dim": udim, "user_group": ug, "item_dim": idim, "item_group": ig,
            "n_users": n, "n_test": n_test,
            model_a: a[valid].mean(), model_b: b[valid].mean(),
            "gap": diff.mean(), "se": se,
            "p": 2 * stats.t.sf(abs(t), df=n - 1),
        })

    cells = pd.DataFrame(rows)
    cells["gap_q"] = stats.false_discovery_control(cells["p"].to_numpy())  # Benjamini-Hochberg
    return cells


def add_interaction(cells: pd.DataFrame) -> pd.DataFrame:
    """Keep the cross cells and add: gap beyond what the row/column marginals predict."""
    is_user_all, is_item_all = cells["user_dim"].eq("all"), cells["item_dim"].eq("all")

    user_gap = cells.loc[is_item_all, ["user_dim", "user_group", "gap"]].rename(columns={"gap": "user_gap"})
    item_gap = cells.loc[is_user_all, ["item_dim", "item_group", "gap"]].rename(columns={"gap": "item_gap"})
    overall_gap = cells.loc[is_user_all & is_item_all, "gap"].iloc[0]

    cross = (
        cells[~is_user_all & ~is_item_all]
        .merge(user_gap, on=["user_dim", "user_group"])
        .merge(item_gap, on=["item_dim", "item_group"])
    )
    cross["expected_gap"] = cross["user_gap"] + cross["item_gap"] - overall_gap
    cross["interaction"] = cross["gap"] - cross["expected_gap"]

    # Approximate: treats the expected gap as fixed. Good enough for screening.
    z = (cross["interaction"] / cross["se"].replace(0, np.nan)).fillna(0)
    cross["interaction_q"] = stats.false_discovery_control(2 * stats.norm.sf(z.abs()))
    return cross


def rank_dimension_pairs(cross: pd.DataFrame, model_a: str, model_b: str, alpha: float = ALPHA) -> pd.DataFrame:
    """Which (user dim x item dim) matrices are worth plotting?"""
    flagged = cross.assign(
        a_wins=(cross["gap_q"] < alpha) & (cross["gap"] > 0),
        b_wins=(cross["gap_q"] < alpha) & (cross["gap"] < 0),
        interacting=cross["interaction_q"] < alpha,
        weighted_abs_interaction=cross["interaction"].abs() * cross["n_test"],
    )
    pairs = flagged.groupby(["user_dim", "item_dim"]).agg(
        n_cells=("gap", "size"),
        a_wins=("a_wins", "sum"),
        b_wins=("b_wins", "sum"),
        n_interacting=("interacting", "sum"),
        _weighted=("weighted_abs_interaction", "sum"),
        _n_test=("n_test", "sum"),
    )
    pairs["mean_abs_interaction"] = pairs.pop("_weighted") / pairs.pop("_n_test")
    pairs["share_interacting"] = pairs["n_interacting"] / pairs["n_cells"]
    pairs["crossover"] = (pairs["a_wins"] > 0) & (pairs["b_wins"] > 0)
    return (
        pairs.rename(columns={"a_wins": f"{model_a}_wins", "b_wins": f"{model_b}_wins"})
        .sort_values(["crossover", "share_interacting"], ascending=False)
    )


def notable_cells(cross: pd.DataFrame, by: str = "interaction", alpha: float = ALPHA, n: int = 15) -> pd.DataFrame:
    """Largest significant cells, ranked by |gap| or |interaction|."""
    significant = cross[cross[f"{'gap' if by == 'gap' else 'interaction'}_q"] < alpha]
    return significant.sort_values(by, key=np.abs, ascending=False).head(n)


def plot_pair(cross: pd.DataFrame, user_dim: str, item_dim: str, model_a: str, model_b: str,
              metric: str = f"recall@{K}", alpha: float = ALPHA) -> None:
    cells = cross[(cross["user_dim"] == user_dim) & (cross["item_dim"] == item_dim)]
    rows, cols = cells["user_group"].unique(), cells["item_group"].unique()  # keeps definition order

    def grid(column: str) -> pd.DataFrame:
        return cells.pivot(index="user_group", columns="item_group", values=column).reindex(index=rows, columns=cols)

    gap = grid("gap")
    labels = gap.round(2).astype(str) + np.where(grid("gap_q") < alpha, "*", "")
    vmin, vmax = min(grid(model_a).min().min(), grid(model_b).min().min()), max(grid(model_a).max().max(), grid(model_b).max().max())

    fig, axes = plt.subplots(1, 3, figsize=(22, 5), constrained_layout=True)
    for ax, model in zip(axes[:2], (model_a, model_b)):
        sns.heatmap(grid(model), annot=True, fmt=".2f", cmap="viridis", vmin=vmin, vmax=vmax, ax=ax)
        ax.set_title(f"{model} {metric}")
    sns.heatmap(gap, annot=labels, fmt="", cmap="RdBu", center=0, ax=axes[2])
    axes[2].set_title(f"{model_a} − {model_b} (blue: {model_a} better, * = FDR q < {alpha})")

In [ ]:
data = build_evaluation_data(df, MODELS)
distance_matrix = BeyondAccuracyMetrics.build_genre_distance_matrix(list(data.items["genres"]), data.relevant.shape[1], jaccard_distance)
user_groups, item_groups = make_user_groups(data), make_item_groups(data)

user_tbl = user_group_table(data, user_groups, distance_matrix)
item_tbl = item_group_table(data, item_groups)

# Screening: all combinations at once
cells = screen_cells(data, user_groups, item_groups, "EASE", "BPR")
cross = add_interaction(cells)

pairs = rank_dimension_pairs(cross, "EASE", "BPR")   # 1. which matrices matter
notable_cells(cross, by="gap")                                  # 2. where one model clearly wins
notable_cells(cross, by="interaction")                          # 3. surprising combinations
plot_pair(cross, "activity", "pop_bucket", "EASE", "BPR")       # 4. plot the top pairs only

AttributeError: type object 'RankingMetrics' has no attribute 'calculate_per_user_metrics'